In [7]:
!pip install spacy streamlit -q
!python -m spacy download en_core_web_md -q
!npm install -g localtunnel -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 27.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [8]:
%%writefile app.py
import streamlit as st
import spacy
@st.cache_resource
def load_nlp():
    return spacy.load("en_core_web_md")
nlp = load_nlp()
FAQ_DATA = [
    {"question": "What is your return policy?", "answer": "You can return any product within 30 days of purchase for a full refund."},
    {"question": "How long does shipping take?", "answer": "Standard shipping takes 3-5 business days."},
    {"question": "Can I change or cancel my order?", "answer": "Orders can be changed or canceled within 1 hour of placement."},
    {"question": "What payment methods do you accept?", "answer": "We accept all major credit cards, PayPal, and Apple Pay."}
]
def preprocess_text(text):
    doc = nlp(text.lower().strip())
    cleaned_tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return nlp(" ".join(cleaned_tokens))
def get_best_faq_match(user_query, faq_list, threshold=0.65):
    user_doc = preprocess_text(user_query)
    if not user_doc.vector_norm:
        return "I didn't quite catch that. Could you please rephrase?"
    best_match, highest_score = None, -1.0
    for faq in faq_list:
        faq_doc = preprocess_text(faq["question"])
        similarity = user_doc.similarity(faq_doc)
        if similarity > highest_score:
            highest_score, best_match = similarity, faq
    return best_match["answer"] if highest_score >= threshold else "I couldn't find a perfect match. Try rephrasing."
st.title(" Colab-Hosted FAQ Bot")
if "messages" not in st.session_state:
    st.session_state.messages = [{"role": "assistant", "content": "Hello! Ask me a question."}]
for message in st.session_state.messages:
    with st.chat_message(message["role"]): st.markdown(message["content"])
if user_input := st.chat_input("Type here..."):
    with st.chat_message("user"): st.markdown(user_input)
    st.session_state.messages.append({"role": "user", "content": user_input})
    bot_response = get_best_faq_match(user_input, FAQ_DATA)
    with st.chat_message("assistant"): st.markdown(bot_response)
    st.session_state.messages.append({"role": "assistant", "content": bot_response})


Writing app.py


In [ ]:

import urllib
print("Your Tunnel Password IP is:", urllib.request.urlopen('https://ident.me').read().decode('utf8'))
!streamlit run app.py & npx localtunnel --port 8501


Your Tunnel Password IP is: 130.211.240.205
⠙⠹⠸⠼⠴⠦

⠧⠇⠏⠋⠙your url is: https://violet-mugs-stick.loca.lt
2026-07-01 06:46:51.572 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://130.211.240.205:8501

